# 0. 환경설정
## import

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

## 함수정의

In [2]:
# def draw_lineChart (df, y_val, title, ylabel, is_rank=False):

#     autorange =  "reversed" if is_rank else True
#     dtick = 1 if is_rank else None
    
#     # 1. Plotly 라인 차트 생성
#     fig = px.line(
#         df, 
#         x='b_date', 
#         y=y_val, 
#         color='movieNm',       # 영화별 색상 구분
#         markers=True,          # 데이터 지점에 점 표시
#         title=title,
#         hover_data=['movieNm', 'b_date', y_val] # 마우스 올렸을 때 출력할 데이터
#     )
    
#     fig.update_layout(
#         yaxis=dict(
#             autorange=autorange,          # Y축 반전 (1위가 최상단)
#             dtick=dtick,                       # Y축 간격을 1 단위로 정수 표시
#             title=f'{ylabel} ({y_val})'
#         ),
#         xaxis=dict(
#             title='날짜 (Date)',
#             tickangle=-45                   # 날짜 라벨 45도 회전
#         ),
#         hovermode="x unified",              # 같은 날짜 위치에 마우스를 올리면 모든 영화 순위를 한 번에 비교
#         legend_title_text='영화 제목',
#         template='plotly_white',            # 배경을 깔끔한 흰색으로 설정
#         width=1000,
#         height=600
#     )

#     # 3. 차트 출력
#     fig.show()

def draw_lineChart (df, y_val, title, ylabel, is_rank=False):

    autorange = "reversed" if is_rank else True
    dtick = 1 if is_rank else None
    
    fig = px.line(
        df, 
        x='b_date', 
        y=y_val, 
        color='movieNm',
        markers=True,
        title=title,
        hover_data=['movieNm', 'b_date', y_val]
    )
    
    fig.update_layout(
        yaxis=dict(
            autorange=autorange,
            dtick=dtick,
            title=f'{ylabel} ({y_val})'
        ),
        xaxis=dict(
            title='날짜 (Date)',
            tickangle=-45
        ),
        # --- [수정 구간] 범례 위치 및 여백 설정 ---
        legend=dict(
            orientation="h",        # 범례를 가로로 배치
            yanchor="top",
            y=-0.25,                # x축 아래로 배치 (값으로 위치 조정 가능)
            xanchor="center",
            x=0.5,                  # 중앙 정렬
            title_text=''           # 범례 제목 제거로 공간 확보
        ),
        margin=dict(b=150),          # 하단 여백을 늘려 범례 잘림 방지
        # ----------------------------------------
        hovermode="x unified",
        template='plotly_white',
        width=1000,
        height=650                  # 범례 공간 확보를 위해 높이 약간 확대
    )

    fig.show()

# 1. 데이터 호출 및 확인

## 데이터 호출

In [3]:
movie_df = pd.read_csv('../data/pre_processed/movie_info_20260724~20260810.csv')
review_df = pd.read_csv('../data/pre_processed/review_20260724~20260810.csv')
master_df = pd.read_csv('../data/pre_processed/movie_master_20260724~20260810.csv')

## 데이터 확인

### review_df : 리뷰 데이터 모음
 * `id` : 영화 ID
 * `reviewer_name` : 리뷰어 이름
 * `score` : 점수
 * `review` : 리뷰 내용

In [4]:
# 데이터 확인
review_df.info()

# 중복값 확인
review_df.duplicated().sum()
# 결측값 확인
review_df.isna().sum()
# 이상치 확인
# score 범위 1 ~ 10점
review_df['score'].min() # 4
review_df['score'].max() # 9

# 필요 전처리 
# id값 텍스트화 시켜야 함 + 끝에 .0 삭제 필요
review_df['id'] = review_df['id'].astype('str').str.split('.').str[0]

review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             48 non-null     int64 
 1   reviewer_name  48 non-null     object
 2   score          48 non-null     int64 
 3   review         48 non-null     object
dtypes: int64(2), object(2)
memory usage: 1.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             48 non-null     object
 1   reviewer_name  48 non-null     object
 2   score          48 non-null     int64 
 3   review         48 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.6+ KB


### master_df : 영화의 정보를 담고, 리뷰와의 연결성을 확보할 수 있는 테이블

* `title` : 영화 제목
* `score` : 영화 평점
* `id` : 영화 id
* `genre` : 영화 장르
* `grade` : 등급
* `time` : 영화 상영시간
* `director` : 감독
* `expert_score` : 전문가 평점
* `nation` : 국가
* `movieCd` : 영화코드 (API 데이터 연결용)

In [5]:
# 데이터 확인
master_df.info()

# 데이터 형 변환
# id -> 문자열 + .0 삭제
master_df['id'] = master_df['id'].astype('str').str.split('.').str[0]
# movieCd -> 문자열
master_df['movieCd'] = master_df['movieCd'].astype('str')

# 중복값 확인
master_df.duplicated().sum()
master_df['movieCd'].duplicated().sum()

# 결측값 확인
master_df.isna().sum()
master_df[master_df['score'].isna()]
# 단 한 건의 영화이고, 해당 영화에 대한 정보를 확보할 수 있으므로
# 수기로 결측 대체 진행

# 영화 코드가 있으므로, 해당 코드를 기반으로 검색 진행
master_df.loc[master_df['movieCd'] == '20264635',
              ['score', 'id', 'genre', 'grade', 'time', 'director', 'expert_score', 'nation']
            ] = [0, 'N/A', '애니메이션', '12세이상관람가', '109분', '하스이 타카히로', 0, '일본']
master_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         12 non-null     object 
 1   score         11 non-null     float64
 2   id            11 non-null     float64
 3   genre         11 non-null     object 
 4   grade         11 non-null     object 
 5   time          11 non-null     object 
 6   director      11 non-null     object 
 7   expert_score  11 non-null     float64
 8   nation        11 non-null     object 
 9   movieCd       12 non-null     int64  
dtypes: float64(3), int64(1), object(6)
memory usage: 1.1+ KB


,title,score,id,genre,grade,time,director,expert_score,nation,movieCd
0,눈동자,5.00,63186,스릴러,15세이상관람가,105분,염지호,5.00,한국,20242402
1,다윗,5.00,63108,애니메이션,전체관람가,109분,필커닝햄,5.00,미국,20262902
2,명탐정 코난: 하이웨이의 타천사,0.00,N/A,애니메이션,12세이상관람가,109분,하스이 타카히로,0.00,일본,20264635
3,모아나,5.67,62818,"어드벤처,액션",전체관람가,115분,토마스케일,5.67,미국,20259946
4,미니언즈 & 몬스터즈,6.50,63032,애니메이션,전체관람가,89분,피에르코팽,6.50,미국,20261784


### movie_df : KOBIS API를 통해 수집한 일별 영화 순위 및 정보

* `rank` : 순위
* `rankInten` : 이전일 대비 순위 증감분
* `rankOldAndNew` : 이전일 대비 랭크 신규 진입 여부 (OLD : 기존 / NEW : 신규)
* `movieCd` : 영화 대표 코드
* `movieNm` : 영화 이름 (국문)
* `openDt` : 영화 개봉일
* `salesAmt` : 해당 날짜의 매출액
* `salesShare` : 해당일자 상영작의 매출총액 대비 해당 영화의 매출비율
* `salesInten` : 전일 대비 매출액 증감분
* `salesChange` : 전일 대비 매출 증감 비율
* `salesAcc` : 누적 매출액
* `audiCnt` : 해당일의 관객수
* `audiInten` : 전일 대비 관객수 증감분
* `audiChange` : 전일 대비 관객수 증감 비율
* `audiAcc` : 누적관객수
* `scrnCnt` : 해당 일자에 상영한 스크린 수 출력
* `showCnt` : 해당 일자에 상영된 횟수 출력

In [6]:
# 데이터 확인
# movie_df.info()

# 데이터 형 변환
# movieCd -> 문자열
master_df['movieCd'] = master_df['movieCd'].astype('str')

# 중복값 확인
movie_df.duplicated().sum()

# 결측값 확인
movie_df.isna().sum()

# 이상치 확인
movie_df.describe()
#  누적 매출액 >= 해당일 매출액
movie_df[movie_df['salesAcc'] < movie_df['salesAmt']]
# 누적 관객수 >= 해당일 관객수
movie_df[movie_df['audiAcc'] < movie_df['audiCnt']]
# 상영 횟수 >= 상영 스크린 수
movie_df[movie_df['showCnt'] < movie_df['scrnCnt']]
# rankOldAndNew : OLD / NEW
movie_df['rankOldAndNew'].unique()

array(['OLD', 'NEW'], dtype=object)

# 2-1. 집계 기간 내 박스오피스 영화 기본 분석

## 집계 기간 내 박스오피스 순위권 내 (10위) 포함된 영화 특징 조사

In [7]:
# 국가
master_df.groupby('nation').count() # 미국(6), 한국(4), 일본(2)

# 장르
master_df.groupby('genre').count()
# 애니메이션(5)
# 다큐멘터리, 스릴러, [액션, 모험, 드라마],[액션, 스릴러, SF], [어드벤처, 액션], 코미디, [판타지, 어드벤처, 액션] (1)

# 등급
master_df.groupby('grade').count()
# 전체관람가 (5)
# 15세이상관람가 (4)
# 12세이상관람가 (3)

# 시간 분포
master_df
# '분' 제거 후 정수형 변환
master_df['time_num'] = master_df['time'].str.replace('분', '').astype(int)

# Plotly 히스토그램
fig = px.histogram(master_df, x='time_num', nbins=10, title='영화 상영시간(분) 분포')
fig.update_layout(xaxis_title='상영시간 (분)', yaxis_title='영화 수')
fig.show()

## 기간에 따른 영화 순위 변화

In [8]:
draw_lineChart(movie_df, 'rank', '영화별 일별 박스오피스 순위 변동 추이', '순위', True)

## 기간에 따른 일일 매출액 변화

In [9]:
draw_lineChart(movie_df, 'salesAmt', '영화별 일별 박스오피스 일일 매출액 변동 추이', '일일 매출액')

## 기간에 따른 일일 관객 수 변화

In [10]:
draw_lineChart(movie_df, 'audiCnt', '영화별 일별 박스오피스 일일 관객 수 변동 추이', '일일 관객 수')

## 기간에 따른 누적 매출액 변화

In [11]:
draw_lineChart(movie_df, 'salesAcc', '영화별 일별 박스오피스 누적 매출 변동 추이', '누적 매출')

## 기간에 따른 누적 관객 수 변화

In [12]:
draw_lineChart(movie_df, 'audiAcc', '영화별 일별 박스오피스 누적 관객 수 변동 추이', '누적 관객 수')

## 기간에 따른 스크린 수

In [13]:
draw_lineChart(movie_df, 'scrnCnt', '영화별 일별 박스오피스 스크린 수 변동 추이', '스크린 수')

## 기간에 따른 상영 수

In [14]:
draw_lineChart(movie_df, 'showCnt', '영화별 일별 박스오피스 상영 수 변동 추이', '상영 수')

* 집계 기간 내 순위권 내 포함된 영화는 총 12개의 영화가 순위권 내에 진입하였다.
  * 국가별 : 미국(6개), 한국(4개), 일본(2개)
  * 카테고리 : 애니메이션 (5개), 액션 포함 (4개)
  * 등급 : 전체관람가 (5개), 15세이상관람가(4개), 12세이상관람가(3개)

* 집계 기간 내 1위에 등극한 영화는 총 3개의 영화가 집계 기간 동안 1위에 등극
  * 호프 (26년 7월 24일 ~ 26년 7월 28일)
  * 스파이더맨: 브랜드 뉴 데이 (26년 7월 29일 ~ 26년 8월 4일)
  * 오디세이 (26년 8월 5일 ~ 26년 8월 9일)

  * 기존 1위를 유지하던 호프가 스파이더맨의 개봉과 함께 1위에서 추락하기 시작하였다.
  * 또한, 스파이더맨: 브랜드 뉴 데이 역시 오디세이의 개봉과 함께 1위를 내어주고 2위를 유지하고 있는 것으로 확인된다.

* 일일 매출액과 일일 관객 수 확인
  * 10위권 내 포함된 영화 중 유의미한 관객 및 매출의 차이를 보이는 것은 1위를 한 영화
  * 일일 매출액과 일일 관객 수는 유사한 패턴을 보이고 있다.
  * 스파이더맨: 브랜드 뉴 데이는 개봉 후 맞이한 첫 주말에 폭발적인 관객 수를 확보할 수 있었다.
  * 오디세이의 경우에는 일일 매출액과 일일 관객 수의 패턴이 약간 상이한 모습을 보이고 있다.
  * 관객 수 대비 매출이 더 상승했다고 해석할 수 있는데, 오디세이의 경우에는 상대적으로 단가가 비싼 아이맥스, 4DX 등으로 더 많이 소비되었을 가능성이 크다.

* 누적 매출액과 누적 관객 수 확인
  * 압도적인 누적 매출액 : 왕과 사는 남자
  * 스파이더맨의 상승세 군체를 뛰어 넘었다.
  * 호프는 실질적 1위 효과에도 불구하고, 누적 매출이 그렇게 크지 않음을 확인할 수 있다.
  호프 개봉 당시 호프와 경쟁할만한 영화가 특별히 없었다고 판단할 수 있다.
  * 그에 비해 스파이더맨과 오디세이는 상승세로 지속적으로 누적 매출 및 관객이 상승할 것으로 판단된다.
  * 왕과 사는 남자의 누적 관객 수와 매출을 뛰어넘을 수 있을지 주목해 볼 필요가 있다.

# 2-2. 집계 기간 내 1위 영화를 중심으로 분석

* 위 분석을 통해 확인할 수 있던 점은 최근 영화 산업은 1위 영화 하나를 중심으로 관객과 매출이 몰리고 있음을 확인할 수 있었다.
* 따라서 집계 기간 내 1위를 경험했던 '호프', '스파이더맨', '오디세이'
* 그리고 가장 높은 매출을 기록한 '왕과 사는 남자'
* 총 4개의 영화에 대한 추가 분석을 진행하고자 한다.
* 왕과 사는 남자는 상대적으로 오래 전 개봉한 영화이므로 API를 활용하여 개봉 시점인 2월 4일에 맞춰
* 약 6달 간의 데이터를 추가적으로 확보하여 함께 살펴보고자 한다.

* 왕과 사는 남자는 누적 매출 163억 2156만원을 달성하며 대한민국 영화 역대 최고의 매출액을 달성했다.

In [15]:
movie_6m_df = pd.read_csv('../data/pre_processed/movie_info_20260124~20260810.csv')

In [16]:
target_movie = ['호프', '스파이더맨: 브랜드 뉴 데이', '오디세이', '왕과 사는 남자']
target_movie_df = movie_6m_df[movie_6m_df['movieNm'].isin(target_movie)]

## 타깃 영화 순위 변화

In [17]:
draw_lineChart(target_movie_df, 'rank', '타깃 영화 순위 변동 추이', '순위', True)

## 타깃 영화 일일 매출액 변화

In [18]:
draw_lineChart(target_movie_df, 'salesAmt', '타깃 영화 일일 매출액 변동 추이', '일일 매출액')

## 타깃 영화 일일 관객 수 변화

In [19]:
draw_lineChart(target_movie_df, 'audiCnt', '타깃 영화 일일 관객 수 변동 추이', '일일 관객 수')

## 타깃 영화 누적 매출액 변화

In [20]:
draw_lineChart(target_movie_df, 'salesAcc', '타깃 영화 누적 매출 변동 추이', '누적 매출')

## 타깃 영화 누적 관객 수 변화

In [21]:
draw_lineChart(target_movie_df, 'audiAcc', '타깃 영화 누적 관객 수 변동 추이', '누적 관객 수')

## 타깃 영화 스크린 수 변화

In [22]:
draw_lineChart(target_movie_df, 'scrnCnt', '타깃 영화 스크린 수 변동 추이', '스크린 수')

## 타깃 영화 상영 수 변화

In [23]:
draw_lineChart(target_movie_df, 'showCnt', '타깃 영화 상영 수 변동 추이', '상영 수')

## 4개 타깃 영화의 주요 특징

In [24]:
target_movie_df[['movieCd', 'movieNm']] # 왕과 사는 남자 - 20242837
master_df.loc[len(master_df)] = {
    'title': '왕과 사는 남자',
    'score': 8.07,
    'id': 62893,
    'genre': '시대극, 드라마',
    'grade': '12세이상관람가',
    'time': '116분',
    'director': '장항준',
    'expert_score': 6.57,
    'nation': '한국',
    'movieCd': '20242837'
}

In [25]:
target_master_df = master_df[master_df['title'].isin(target_movie)]
target_master_df

# 크롤링 데이터가 score와 expert_score가 같은 별점을 나타내고 있음을 확인.
# 전문가의 별점이 아닌 관객들의 별점을 추가로 확인 진행.
# 수기로 수정 진행
target_master_df.loc[target_master_df['title'] == '왕과 사는 남자', 'score'] = 6.57
target_master_df['audi_score'] = [7.33, 7.90, 5.88, 8.07]

/var/folders/1w/__xlnm915tg4msbkc4r6jv_r0000gn/T/ipykernel_23196/3096363277.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_master_df['audi_score'] = [7.33, 7.90, 5.88, 8.07]


In [26]:
target_master_df

,title,score,id,genre,grade,time,director,expert_score,nation,movieCd,time_num,audi_score
6,스파이더맨: 브랜드 뉴 데이,7.33,63091,"판타지,어드벤처,액션",12세이상관람가,144분,데스틴크리튼,7.33,미국,20262770,144.0,7.33
8,오디세이,7.55,62585,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,20250654,172.0,7.90
11,호프,6.83,62480,"액션,스릴러,SF",15세이상관람가,156분,나홍진,6.83,한국,20233219,156.0,5.88
12,왕과 사는 남자,6.57,62893,"시대극, 드라마",12세이상관람가,116분,장항준,6.57,한국,20242837,NaN,8.07


In [27]:
display(target_movie_df[target_movie_df['movieNm'] == '왕과 사는 남자'].groupby('rank').count())

,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,b_date,c_date
rank,,,,,,,,,,,,,,,,,,
1,59,59,59,59,59,59,59,59,59,59,59,59,59,59,59,59,59,59
2,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
3,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11
4,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
6,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13
7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7
9,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
10,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7


* 왕과 사는 남자는 누적 매출 163억 2156만원을 달성하며 대한민국 영화 역대 최고의 매출액을 달성했다.
* 그 배경에는 하루 반짝 하는 일일 매출액 보다 무려 59일 동안 1위를 지켜온 힘에 있다고 볼 수 있다.

In [28]:
print('호프')
print(len(target_movie_df[
    (target_movie_df['movieNm'] == '호프') & 
    (target_movie_df['rank'] == 1)
]))

print('스파이더맨: 브랜드 뉴 데이')
print(len(target_movie_df[
    (target_movie_df['movieNm'] == '스파이더맨: 브랜드 뉴 데이') & 
    (target_movie_df['rank'] == 1)
]))

print('오디세이')
print(len(target_movie_df[
    (target_movie_df['movieNm'] == '오디세이') & 
    (target_movie_df['rank'] == 1)
]))

호프
14
스파이더맨: 브랜드 뉴 데이
7
오디세이
6


* 호프 역시 14일 1위를 기록하는 등의 성과를 보이지만
* 일일 매출 및 방문자 수가 저조하여, 결과론적으로 유의미한 성공을 거두었다고 보기 어렵다.
* 스파이더맨은 개봉 직후 급격한 상승등을 보이면 좋은 흐름을 보이지만
* 오디세이의 등장과 함께 2위로 밀려난 형태를 띄고 있다.
* 다만, 오디세이와 스파이더맨은 개봉이 얼마 되지 않았고
* 현재 꾸준하게 관람객들이 선택하며 좋은 흐름을 보여주고 있다.

# 2-3. 왕과 사는 남자 개봉 당시 경쟁작의 유무 파악

* 왕과 사는 남자가 영화관에 상영되는 기간동안 있던 영화 파악

In [29]:
# 1. 대상 영화('왕과 사는 남자')의 시작일과 종료일 자동 추출
target_movie = '왕과 사는 남자'
start_date = movie_6m_df[movie_6m_df['movieNm'] == target_movie]['b_date'].min()
end_date = movie_6m_df[movie_6m_df['movieNm'] == target_movie]['b_date'].max()

# 2. 해당 기간 데이터 필터링
target_period_df = movie_6m_df[
    (movie_6m_df['b_date'] >= start_date) & 
    (movie_6m_df['b_date'] <= end_date)
]

# 3. [추가] 해당 기간 중 rank가 1위인 적이 한 번이라도 있었던 영화 목록 추출
ranked_first_movies = target_period_df[target_period_df['rank'] == 1]['movieNm'].unique()

# 4. 1위를 했던 영화들의 데이터만 최종 필터링
target_period_df = target_period_df[target_period_df['movieNm'].isin(ranked_first_movies)]

## 상영 기간 순위 변화

In [30]:
draw_lineChart(target_period_df, 'rank', '상영기간 순위 변동 추이', '순위', True)

## 상영 기간 일일 매출액 변화

In [31]:
draw_lineChart(target_period_df, 'salesAmt', '상영 기간 일일 매출액 변동 추이', '일일 매출액')

## 상영 기간 누적 매출액 변화

In [32]:
draw_lineChart(target_period_df, 'salesAcc', '상영기간 누적 매출 변동 추이', '누적 매출')

* 왕과 사는 남자의 1위 등극 기간이 길어지고, 순위가 하락하는 시점에 다른 영화들이 개봉하며
* 일일 매출과 관객 수 변화 혹은 순위에 변화가 있었다.
* 하지만 순위 하락이 매우 단계적으로 내려가며, 내려간 기간에도 순위를 유지하는 힘이 강했기에
* 단순하게 상영 당시 주요 경쟁작이 없어서 발생한 결과로만 단정짓기에는 어렵다.

## 2-4. 오디세이와 스파이더맨의 예상 매출 추이 및 관객 수 추이 예상

In [33]:
movie_6m_df

,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,b_date,c_date
0,1,0,OLD,20249255,만약에 우리,2025-12-31,1079629440,28.7,577370770,115.0,18622648660,108141,58256,116.8,1904759,1047,3929,2026-01-24,2026-08-11 11:44:21
1,2,0,OLD,20247457,신의악단,2025-12-31,569067940,15.1,311215040,120.7,5874083030,57760,31328,118.5,612334,811,1989,2026-01-24,2026-08-11 11:44:21
2,3,0,OLD,20256396,아바타: 불과 재,2025-12-17,648875900,17.3,377467760,139.1,76113150250,50122,29432,142.3,6520012,772,1822,2026-01-24,2026-08-11 11:44:21
3,4,3,OLD,20250482,"신비아파트 10주년 극장판: 한 번 더, 소환",2026-01-14,231017080,6.1,176114080,320.8,1939440580,25729,19550,316.4,217242,683,1340,2026-01-24,2026-08-11 11:44:21
4,5,-1,OLD,20249624,프로젝트 Y,2026-01-21,256604780,6.8,114919240,81.1,781367510,24718,10002,68.0,80420,800,2268,2026-01-24,2026-08-11 11:44:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1985,6,0,OLD,20261784,미니언즈 & 몬스터즈,2026-07-15,19237500,0.4,-25821180,-57.3,7612121820,2193,-2821,-56.3,765485,208,217,2026-08-10,2026-08-11 11:42:25
1986,7,3,OLD,20259781,토이 스토리 5,2026-06-17,10804150,0.2,-7973700,-42.5,30219382390,1112,-795,-41.7,2940488,109,112,2026-08-10,2026-08-11 11:42:25
1987,8,-3,OLD,20262902,다윗,2026-07-10,9168150,0.2,-69105900,-88.3,2816467330,958,-7384,-88.5,290308,65,69,2026-08-10,2026-08-11 11:42:25
1988,9,3,OLD,20259946,모아나,2026-07-08,7290500,0.1,748300,11.4,10828554930,826,149,22.0,1021761,67,70,2026-08-10,2026-08-11 11:42:25


In [34]:
# openDt 중 하나라도 날짜 변환이 안 되는 행 찾기
bad_open_dt = pd.to_datetime(movie_6m_df['openDt'], errors='coerce').isna()

# 둘 중 하나라도 문제가 있는 행 출력
display(movie_6m_df[bad_open_dt])

,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,b_date,c_date
1378,9,0,NEW,20264226,쏜애플 '나의 세기' 익스텐디드 플레이 필름,,24000000,1.3,24000000,100.0,24000000,2000,2000,100.0,2000,29,60,2026-06-10,2026-08-11 11:42:59
1416,7,0,NEW,20264331,디어 마이 히어로,,27500000,0.8,27500000,100.0,27500000,5500,5500,100.0,5500,120,328,2026-06-14,2026-08-11 11:42:56
1428,9,-2,OLD,20264331,디어 마이 히어로,,8790000,0.9,-18710000,-68.0,36290000,1758,-3742,-68.0,7258,110,279,2026-06-15,2026-08-11 11:42:56
1437,8,1,OLD,20264331,디어 마이 히어로,,7305000,0.7,-1485000,-16.9,43595000,1461,-297,-16.9,8719,113,279,2026-06-16,2026-08-11 11:42:55
1489,10,3,OLD,20264331,디어 마이 히어로,,12375000,0.3,4860000,64.7,78610000,2475,972,64.7,15722,153,185,2026-06-21,2026-08-11 11:42:52
1729,10,0,NEW,20261421,나무의 노래,,9297000,0.2,9297000,100.0,36505000,1033,1033,100.0,4185,6,6,2026-07-15,2026-08-11 11:42:39


In [35]:
movie_6m_df.loc[movie_6m_df['movieNm'] == "쏜애플 '나의 세기' 익스텐디드 플레이 필름", 'openDt'] = '2026-06-10'
movie_6m_df.loc[movie_6m_df['movieNm'] == "디어 마이 히어로", 'openDt'] = '2026-06-14'

# 나무의 노래는 검색이 되지 않고, 9월 개봉 예정이라고만 나와서 삭제 진행.
movie_6m_df = movie_6m_df[movie_6m_df['movieNm'] != '나무의 노래']

In [36]:
import pandas as pd
import numpy as np

# 1. 날짜 타입 변환 및 정렬
movie_6m_df['b_date'] = pd.to_datetime(movie_6m_df['b_date'])
movie_6m_df['openDt'] = pd.to_datetime(movie_6m_df['openDt'])
df = movie_6m_df.sort_values(['movieCd', 'b_date']).reset_index(drop=True)

# 2. 핵심 상대 변수: 개봉 후 경과일 (days_since_release)
df['days_since_release'] = (df['b_date'] - df['openDt']).dt.days + 1

# 3. 달력/시즌 변수
df['dayofweek'] = df['b_date'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

# 4. 범주형 변수 수치화
df['is_new'] = (df['rankOldAndNew'] == 'NEW').astype(int)

# 5. [중요] 시계열 Lag 변수 생성 (전날 데이터)
# 오늘(t)의 관객수 예측을 위해 전날(t-1), 전전날(t-2) 관객수와 스크린 수를 가져옴
for lag in [1, 2]:
    df[f'prev_{lag}d_audi'] = df.groupby('movieCd')['audiCnt'].shift(lag)
    df[f'prev_{lag}d_scrn'] = df.groupby('movieCd')['scrnCnt'].shift(lag)
    df[f'prev_{lag}d_rank'] = df.groupby('movieCd')['rank'].shift(lag)

# Lag 변수로 인해 발생한 초기 개봉 1~2일차 결측치(NaN)는 0으로 채우거나 제거
df_model = df.dropna(subset=['prev_1d_audi']).copy()

/var/folders/1w/__xlnm915tg4msbkc4r6jv_r0000gn/T/ipykernel_23196/814265563.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_6m_df['b_date'] = pd.to_datetime(movie_6m_df['b_date'])
/var/folders/1w/__xlnm915tg4msbkc4r6jv_r0000gn/T/ipykernel_23196/814265563.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_6m_df['openDt'] = pd.to_datetime(movie_6m_df['openDt'])


In [40]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# 사용할 독립변수(X)와 종속변수(y) 정의
feature_cols = [
    'days_since_release', 'dayofweek', 'is_weekend', 'is_new',
    'scrnCnt', 'showCnt',                  # 당일 스크린/상영 횟수
    'prev_1d_audi', 'prev_2d_audi',        # 과거 관객수 추이
    'prev_1d_scrn', 'prev_1d_rank',        # 과거 스크린/순위 추이
    'audiAcc', 'salesAcc'                  # 누적 수치
]
target = 'audiCnt'

X = df_model[feature_cols]
y = df_model[target]

# 시계열 분리 (특정 날짜 이전: Train / 이후: Test)
split_date = '2026-06-01'
train_mask = df_model['b_date'] < split_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

# 모델 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 예측 및 성능 평가
y_pred = model.predict(X_test)

print(f"MAE (평균 오차 관객수): {mean_absolute_error(y_test, y_pred):,.0f}명")
print(f"MAPE (평균 오차율): {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
master_df

,title,score,id,genre,grade,time,director,expert_score,nation,movieCd,time_num
0,눈동자,5.00,63186,스릴러,15세이상관람가,105분,염지호,5.00,한국,20242402,105.0
1,다윗,5.00,63108,애니메이션,전체관람가,109분,필커닝햄,5.00,미국,20262902,109.0
2,명탐정 코난: 하이웨이의 타천사,0.00,N/A,애니메이션,12세이상관람가,109분,하스이 타카히로,0.00,일본,20264635,109.0
3,모아나,5.67,62818,"어드벤처,액션",전체관람가,115분,토마스케일,5.67,미국,20259946,115.0
4,미니언즈 & 몬스터즈,6.50,63032,애니메이션,전체관람가,89분,피에르코팽,6.50,미국,20261784,89.0
5,사랑의 하츄핑: 고래보석의 전설,7.00,63075,애니메이션,전체관람가,105분,김수훈,7.00,한국,20262381,105.0
6,스파이더맨: 브랜드 뉴 데이,7.33,63091,"판타지,어드벤처,액션",12세이상관람가,144분,데스틴크리튼,7.33,미국,20262770,144.0
7,어떻게 해야 했을까?,6.50,63282,다큐멘터리,12세이상관람가,101분,후지노토모아키,6.50,일본,20264148,101.0
8,오디세이,7.55,62585,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,20250654,172.0
9,오케이 마담2,5.00,63240,코미디,15세이상관람가,108분,이철하,5.00,한국,20255484,108.0


In [ ]:
movie_6m_df

,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,b_date,c_date
0,1,0,OLD,20249255,만약에 우리,2025-12-31,1079629440,28.7,577370770,115.0,18622648660,108141,58256,116.8,1904759,1047,3929,2026-01-24,2026-08-11 11:44:21
1,2,0,OLD,20247457,신의악단,2025-12-31,569067940,15.1,311215040,120.7,5874083030,57760,31328,118.5,612334,811,1989,2026-01-24,2026-08-11 11:44:21
2,3,0,OLD,20256396,아바타: 불과 재,2025-12-17,648875900,17.3,377467760,139.1,76113150250,50122,29432,142.3,6520012,772,1822,2026-01-24,2026-08-11 11:44:21
3,4,3,OLD,20250482,"신비아파트 10주년 극장판: 한 번 더, 소환",2026-01-14,231017080,6.1,176114080,320.8,1939440580,25729,19550,316.4,217242,683,1340,2026-01-24,2026-08-11 11:44:21
4,5,-1,OLD,20249624,프로젝트 Y,2026-01-21,256604780,6.8,114919240,81.1,781367510,24718,10002,68.0,80420,800,2268,2026-01-24,2026-08-11 11:44:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1985,6,0,OLD,20261784,미니언즈 & 몬스터즈,2026-07-15,19237500,0.4,-25821180,-57.3,7612121820,2193,-2821,-56.3,765485,208,217,2026-08-10,2026-08-11 11:42:25
1986,7,3,OLD,20259781,토이 스토리 5,2026-06-17,10804150,0.2,-7973700,-42.5,30219382390,1112,-795,-41.7,2940488,109,112,2026-08-10,2026-08-11 11:42:25
1987,8,-3,OLD,20262902,다윗,2026-07-10,9168150,0.2,-69105900,-88.3,2816467330,958,-7384,-88.5,290308,65,69,2026-08-10,2026-08-11 11:42:25
1988,9,3,OLD,20259946,모아나,2026-07-08,7290500,0.1,748300,11.4,10828554930,826,149,22.0,1021761,67,70,2026-08-10,2026-08-11 11:42:25
